In [1]:
from langchain_openai import ChatOpenAI
import os
from pydantic import SecretStr

llm = ChatOpenAI(
    model="gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=SecretStr(os.environ["OPENROUTER_API_KEY"])
)

## Chain with Custom Runnable

In [2]:
#Task 1
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie summarizer. Summarize the following text in 2-3 sentences."),
    ("user", "Please summarize the following movie: {text}"),
])

In [3]:
#Task 2
llm_openai = ChatOpenAI(
    model="gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=SecretStr(os.environ["OPENROUTER_API_KEY"]),
    temperature=0.7
)

In [4]:
#Task 3
from langchain_core.output_parsers import StrOutputParser

str_parser = StrOutputParser()

In [7]:
#Task 4
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text: str) -> dict:
    return {"text": text}

dict_maker_runnable = RunnableLambda(dictionary_maker)

## Parallel Chain 1

In [8]:
#Task 1
linkedin_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator. Generate a LinkedIn post based on the following text."),
    ("user", "Please generate a LinkedIn post for the following text: {text}"),
])

#Task 2

llm_openai_linkedin = ChatOpenAI(
    model="gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=SecretStr(os.environ["OPENROUTER_API_KEY"]),
    temperature=0.7
)

#Task 3
linkedin_str_parser = StrOutputParser()

#Chain for LinkedIn post generation
chain_linkedin = linkedin_prompt_template | llm_openai_linkedin | linkedin_str_parser

## Parallel Chain 2

In [10]:
def instagram_chain(text: dict):

    text = text["text"]

    #Task 1
    instagram_prompt_template = ChatPromptTemplate.from_messages([
        ("system", "You are an Instagram post generator. Generate an Instagram post based on the following text."),
        ("user", "Please generate an Instagram post for the following text: {text}"),
    ])

    #Task 2
    llm_openai_instagram = ChatOpenAI(
        model="gpt-4o-mini",
        base_url="https://openrouter.ai/api/v1",
        api_key=SecretStr(os.environ["OPENROUTER_API_KEY"]),
        temperature=0.7
    )

    #Task 3
    instagram_str_parser = StrOutputParser()

    #Chain for Instagram post generation
    chain_instagram = instagram_prompt_template | llm_openai_instagram | instagram_str_parser

    chain_instagram_output = chain_instagram.invoke({"text": text})

    return chain_instagram_output

instagram_chain_runnable = RunnableLambda(instagram_chain)

## Final Orchestration

In [ ]:
from langchain_core.runnables import RunnableParallel

final_chain = (
    prompt_template
    | llm_openai
    | str_parser
    | dict_maker_runnable
    | RunnableParallel({
        "linkedin": chain_linkedin,
        "instagram": instagram_chain_runnable,
    })
)

In [ ]:
final_chain.invoke({"text": "Inception"})

{'linkedin': '🌌✨ Exploring the Depths of the Mind: Lessons from "Inception" ✨🌌\n\nHave you ever pondered the power of our subconscious? The film "Inception" masterfully takes us on a thrilling journey with Dom Cobb, a skilled thief who extracts secrets from dreams. However, this time, he\'s faced with an intriguing challenge: planting an idea instead of stealing one.\n\nAs Cobb assembles a team to navigate complex layers of dreams, they encounter not just external obstacles, but also the shadows of his haunting past. This blend of action and psychological drama invites us to reflect on profound themes of reality and the immense power of the mind.\n\nIn our professional lives, just like in "Inception," we often confront challenges that require innovative thinking and teamwork. Whether we\'re brainstorming new ideas or tackling complex projects, the lessons from Cobb\'s journey remind us of the importance of collaboration, creativity, and resilience in the face of adversity.\n\nLet\'s em

## Chain as Runnable

In [13]:
def beautify(final_response: dict) -> str:
    linkedin_post = final_response["linkedin"]
    instagram_post = final_response["instagram"]

    return f"LinkedIn Post:\n{linkedin_post}\n\nInstagram Post:\n{instagram_post}"  

beautify_runnable = RunnableLambda(beautify)

beautified_output = final_chain | beautify_runnable

In [14]:
beautified_output.invoke({"text": "Inception"})

'LinkedIn Post:\n🌌✨ The Power of Ideas: Lessons from "Inception" ✨🌌\n\nHave you ever found yourself in a situation where the boundary between reality and dreams becomes blurred? The movie "Inception" brilliantly explores this concept, showcasing the journey of Dom Cobb, a skilled thief who extracts secrets from people\'s subconscious during their dreams. \n\nCobb\'s latest challenge? Planting an idea in a target\'s mind—a task that requires not just skill, but a deep understanding of the human psyche. As he assembles a diverse team to navigate through complex dream layers, they face not only external obstacles but also threats from within their own subconscious.\n\nThis high-stakes quest serves as a powerful reminder of the importance of collaboration, creativity, and resilience. Just like in our professional lives, navigating challenges often requires us to venture into the unknown and confront our fears head-on. \n\nLet\'s take inspiration from Cobb\'s journey and embrace the complex